# MLflow on Colab: track, register and browse a model

This notebook trains a small model and tracks it with MLflow, as [`week11-mlflow.py`](week11-mlflow.py) does:
it logs a run, registers the model as a new version of `iris-classifier`, points the `@champion` alias at that
version and loads the model back by its alias. Then it opens the MLflow UI.

- **On Colab** the UI runs inside the Colab machine, so an [ngrok](https://ngrok.com) tunnel gives it a public
  address. Get an authtoken at <https://dashboard.ngrok.com/get-started/your-authtoken> and store it as the Colab
  secret `NGROK_AUTH_TOKEN` (the key icon in the left bar), or put `NGROK_AUTH_TOKEN=...` in a `.env` file.
  Anyone with the tunnel's address can open the UI, so stop it when you are done (last cell).
- **On your own machine or in the course container** there is no tunnel: the UI is at <http://127.0.0.1:5000>.
  With `COURSE_SMOKE=1` (the course's smoke test) the notebook checks that the UI lists the model, then stops it.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: install what the course container already has
    %pip install -q mlflow==3.18.0 pyngrok==9.1.4 python-dotenv==1.2.4

## Track, register and load the model

The same code as `week11-mlflow.py`.

In [ ]:
import mlflow
from mlflow import MlflowClient
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri("sqlite:///mlflow.db")  # a local database for the runs and the registry
mlflow.set_experiment("iris")

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# skops saves the model and loads back only the types listed as trusted: here, a forest's trees
TRUSTED = ["sklearn.tree._tree.Tree"]

params = {"n_estimators": 10, "random_state": 42}
with mlflow.start_run():
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy)
    # Save with skops, then register a new version
    info = mlflow.sklearn.log_model(
        model, name="model",
        input_example=X_train[:3],
        registered_model_name="iris-classifier",
        skops_trusted_types=TRUSTED)
# Point @champion at this version, load by alias
version = info.registered_model_version
MlflowClient().set_registered_model_alias(
    "iris-classifier", "champion", version)
uri = "models:/iris-classifier@champion"
champion = mlflow.sklearn.load_model(uri)
print("Version", version, champion.score(X_test, y_test))

## Open the MLflow UI

The UI reads the same `mlflow.db`. Open **Models** to see `iris-classifier`, its versions and the `@champion`
alias, and **Experiments > iris** for the runs, their parameters and the accuracy. Run the notebook again to add a
version: the alias moves to it.

In [ ]:
import json
import os
import subprocess
import time
import urllib.request
from urllib.parse import urlparse

PORT = 5000
on_colab = "google.colab" in sys.modules
smoke = os.environ.get("COURSE_SMOKE") == "1"


def ngrok_token():
    """The ngrok authtoken from a Colab secret, the environment or a .env file; None if there is none."""
    if on_colab:
        from google.colab import userdata
        try:
            return userdata.get("NGROK_AUTH_TOKEN")
        except Exception:  # no such secret, or the notebook was not given access to it
            pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv("NGROK_AUTH_TOKEN")


# The UI accepts requests only for the host names it is told about (a defence against DNS rebinding)
allowed_hosts, origins = ["localhost:*", "127.0.0.1:*"], []
token = None if smoke else ngrok_token()
if token:
    from pyngrok import ngrok
    ngrok.kill()  # close tunnels left by an earlier run
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(str(PORT), "http")  # an HTTPS address that forwards to the UI's port
    allowed_hosts.append(urlparse(tunnel.public_url).hostname)
    origins.append(tunnel.public_url)

command = ["mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db", "--port", str(PORT),
           "--allowed-hosts", ",".join(allowed_hosts)]
if origins:
    command += ["--cors-allowed-origins", ",".join(origins)]
ui = subprocess.Popen(command)  # runs in the background until ui.terminate()


def get(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=5) as response:
        return response.read()


for _ in range(120):  # the first start creates the database tables, which takes a few seconds
    try:
        get("/health")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("the MLflow UI did not start")

models = json.loads(get("/ajax-api/2.0/mlflow/registered-models/search"))["registered_models"]
print("Registered models:", ", ".join(f"{m['name']} {m.get('aliases', [])}" for m in models))
if token:
    print("MLflow UI:", tunnel.public_url)
elif on_colab:
    print("No NGROK_AUTH_TOKEN, so no tunnel: see the first cell")
else:
    print(f"MLflow UI: http://127.0.0.1:{PORT}")

## Stop the UI

Run this when you are done, so the tunnel and the server close.

In [ ]:
if token:
    ngrok.kill()
ui.terminate()
ui.wait(timeout=30)
print("MLflow UI stopped")